In [0]:
print("spark is working")

In [0]:
spark.version

In [0]:
data = [
    (101, "Avinash","Lucknow",80000),
    (102,"Rahul","Noida",90000),
    (103,"Priya","Delhi",70000),
    (104,"Rajesh","Noida",60000)
]
columns = ['customer_id','Name','City','Salary']
df  =  spark.createDataFrame(data,columns)
display(df)

In [0]:
# Write the data in bronze
df.write.format("delta").mode("overwrite").saveAsTable("bronze_emp")
# formate("delta") it stores the data in the delta table.
#mode("overritre") replace the table is alredy exist.
#saveAsTable creates a table we can using SQL.

In [0]:
%sql
SELECT *
FROM bronze_emp;

In [0]:
%sql
SELECT COUNT(*) AS TOTAL_RECORDS
FROM bronze_emp;

In [0]:
# Read and Transform the Bronze read and transform the table

silver_df = (
    spark.table("bronze_emp")
    .filter("salary > 70000")
    .withColumnRenamed("name","customer_name")
)
display(silver_df)

In [0]:
# Save our silver file
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_emp")

In [0]:
%sql
SELECT *
FROM silver_emp;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_city_sales AS 
SELECT City,count(*) AS total_customers,
sum(Salary) AS total_sales,
avg(Salary) AS avg_salary
FROM silver_emp
GROUP BY City;

In [0]:
%sql
SELECT *
FROM gold_city_sales
ORDER BY city;

In [0]:
updates = [
    (101,"Avinash","Noida",250000),
    (103,"Priya","Delhi",300000)
]
update_df = spark.createDataFrame(
    updates,
    ["customer_id","customer_name","city","salary"]
)
display(update_df)

In [0]:
update_df.createOrReplaceTempView("updates")

In [0]:
%sql
MERGE INTO silver_emp AS target
USING updates AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
  UPDATE SET
    target.customer_name = source.customer_name,
    target.city = source.city,
    target.salary = source.salary

WHEN NOT MATCHED THEN
  INSERT (
    customer_id,
    customer_name,
    city,
    salary
  )
  VALUES (
    source.customer_id,
    source.customer_name,
    source.city,
    source.salary
  );

In [0]:
%sql
SELECT *
FROM silver_emp
ORDER BY customer_id;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_city_sales AS
SELECT
    city,
    COUNT(*) AS total_customers,
    SUM(Salary) AS total_sales,
    AVG(Salary) AS average_sales
FROM silver_emp
GROUP BY city;

In [0]:
%sql
SELECT *
FROM gold_city_sales
ORDER BY city;

In [0]:
%sql
SELECT *
FROM gold_city_sales;

In [0]:
%sql
SELECT *
FROM silver_emp;

In [0]:
%sql
SELECT *
FROM bronze_emp;